# Example: Run RAG Pipeline

**Prerequisites**: Run `local_colab_setup.ipynb` first!

This notebook demonstrates how to run the RAG pipeline after setup.

---

## Configuration

In [ ]:
# Choose your mode and dataset
MODE = "full"          # "full" = build index, "quick" = load existing index
DATASET = "hospital"   # "hospital" or "corruption"

## Load Config

In [ ]:
from config.config import load_config

config = load_config(f'experiments/{DATASET}_base.yaml')
print(f"✅ Loaded config for {DATASET}")
print(f"   Embedding model: {config.embedding_model}")
print(f"   Use reranker: {config.use_reranker}")

## Option 1: Build Index (First Time or Full Mode)

In [ ]:
if MODE == "full":
    print("🔨 Building index from scratch...")
    
    # Load data
    from data.loader import DataLoader
    loader = DataLoader(config)
    documents = loader.load()
    print(f"✅ Loaded {len(documents)} documents")
    
    # Build FAISS index
    from retrieval.indexer import FAISSIndexer
    indexer = FAISSIndexer(config)
    index = indexer.build_index(documents)
    print(f"✅ Built index with {index.ntotal} vectors")
    
    # Save to Drive (auto-persists)
    index_path = f"outputs/indexes/{DATASET}"
    indexer.save_index(index_path)
    print(f"✅ Index saved to {index_path} (in Google Drive)")
    
    retriever_index_path = index_path
else:
    print("⚡ Quick mode: loading existing index...")
    retriever_index_path = f"outputs/indexes/{DATASET}"

## Initialize Retriever

In [ ]:
from retrieval.retriever import HybridRetriever
from retrieval.reranker import CrossEncoderReranker

# Initialize retriever
retriever = HybridRetriever(config, index_path=retriever_index_path)
print("✅ Retriever initialized")

# Initialize reranker (if enabled)
if config.use_reranker:
    reranker = CrossEncoderReranker(config)
    print("✅ Reranker initialized")
else:
    reranker = None
    print("⚠️  Reranker disabled")

## Initialize RAG Generator

In [ ]:
from generation.rag_generator import AzureRAGGenerator

generator = AzureRAGGenerator(config)
print("✅ Generator initialized")

## Test Query

In [ ]:
# Try a test query
test_query = "What are the main issues discussed?"  # Change this!

print(f"🔍 Query: {test_query}\n")

# Retrieve documents
retrieved_docs = retriever.retrieve(test_query, top_k=10, use_rerank=True)
print(f"✅ Retrieved {len(retrieved_docs)} documents\n")

# Display top 3 results
print("📄 Top 3 Results:")
for i, doc in enumerate(retrieved_docs[:3], 1):
    print(f"\n{i}. Score: {doc['score']:.4f}")
    print(f"   Thread: {doc['thread_id']}")
    print(f"   Content: {doc['content'][:200]}...")

# Generate answer
print("\n🤖 Generating answer...\n")
answer = generator.generate(test_query, retrieved_docs)
print(f"Answer:\n{answer}")

## Launch Gradio Demo (Optional)

In [ ]:
from app.gradio_app import create_demo

demo = create_demo(retriever, reranker, generator, config)
demo.launch(share=True)

# Will output a public URL like: https://xxxxx.gradio.live

---

## Next Steps

1. **Modify code locally** (e.g., edit `src/retrieval/retriever.py`)
2. **Commit and push**: `git add . && git commit -m "..." && git push`
3. **Pull in Colab**: `!cd /content/RAG-embedding && git pull`
4. **Restart kernel** and re-run from top

**Or use MODE="quick"** to skip index building (uses saved index from Drive)